In [1]:
import os

# Prepend the folder containing cdo to PATH
os.environ["PATH"] = "/sw/spack-levante/cdo-2.2.2-4z4icb/bin:" + os.environ["PATH"]

from cdo import Cdo
cdo = Cdo()
print(cdo.version())


2.2.2


In [2]:
import os
import xarray as xr
import pandas as pd
from cdo import Cdo
from joblib import Parallel, delayed

# Paths
data_path_ml = "/pool/data/ERA5/E5/ml/an/1D/"
data_path_pl = "/pool/data/ERA5/E5/pl/an/1D/"
day_path_sf  = "/pool/data/ERA5/E5/sf/an/1D/"

hour_path_sf = "/pool/data/ERA5/E5/sf/an/1H/"
hour_path_pl = "/pool/data/ERA5/E5/pl/an/1H/"
hour_path_ml = "/pool/data/ERA5/E5/ml/an/1H/"

scratch_path = "/scratch/u/u301827/full_midlatitude/"
final_path   = "/work/uc1275/u301827/02_MSE/full_midlatitude/raw/"

os.makedirs(scratch_path, exist_ok=True)
os.makedirs(final_path,   exist_ok=True)

lon_min, lon_max = 0, 360
lat_min, lat_max = 40, 65



In [3]:
def compute_monthly_tasmax_paris_region(
    year, month, hour_path_sf, final_path,
    lon_min, lon_max, lat_min, lat_max,
):
    """
    Compute daily tasmax from hourly ERA5 T2M over a Paris region.
    Aggregates daily tasmax into one monthly NetCDF.

    Parameters
    ----------
    spatial_mean : bool
        If True, average tasmax over the region.
        If False, keep full spatial field.
    """

    var_scratch = os.path.join(scratch_path, "tasmax")
    var_final   = os.path.join(final_path, "tasmax")
    var_scratch_tmp = os.path.join(var_scratch, "temp/")
    os.makedirs(var_scratch, exist_ok=True)
    os.makedirs(var_final,   exist_ok=True)
    os.makedirs(var_scratch_tmp, exist_ok=True)
    
    cdo = Cdo()

    monthly_file = os.path.join(
        final_path, f"tasmax_{year}-{month:02d}_midlatitudes.nc"
    )

    month_str = f"{year}-{month:02d}"

    # Robust month handling
    date_index = pd.date_range(
        start=f"{month_str}-01",
        end=pd.Timestamp(f"{month_str}-01") + pd.offsets.MonthEnd(1)
    )

    daily_datasets = []

    for day in date_index:
        date_str = day.strftime("%Y-%m-%d")
        t2m_file = f"{hour_path_sf}167/E5sf00_1H_{date_str}_167.grb"

        if not os.path.exists(t2m_file):
            print(f"Skipping missing: {t2m_file}")
            continue

        reg_name   = f"reg_t2m_{date_str}.nc"
        box_name   = f"box_t2m_{date_str}.nc"
        reg_file = os.path.join(var_scratch_tmp, reg_name)
        box_file = os.path.join(var_scratch_tmp, box_name)

        # GRIB → regular grid NetCDF
        cdo.setgridtype(
            "regular",
            input=t2m_file,
            output=reg_file,
            options="-f nc --eccodes"
        )

        # Select lon/lat box
        cdo.sellonlatbox(
            lon_min, lon_max, lat_min, lat_max,
            input=reg_file,
            output=box_file
        )

        # Compute daily tasmax
        ds = xr.open_dataset(box_file)

        # Compute daily tasmax AND hour of tasmax
        t2m = ds["2t"]
        
        # Index of maximum temperature in time
        imax = t2m.argmax(dim="time")
        
        # Maximum temperature
        tasmax = t2m.isel(time=imax).rename("tasmax")
        
        # Hour of maximum (UTC)
        tasmax_hour = t2m["time"].isel(time=imax).dt.hour
        tasmax_hour = tasmax_hour.rename("tasmax_hour")
        
        # Add daily time coordinate
        # Drop existing time coordinate (hour of max)
        tasmax = tasmax.reset_coords("time", drop=True)
        tasmax_hour = tasmax_hour.reset_coords("time", drop=True)
        
        # Add daily time dimension
        tasmax = tasmax.expand_dims(time=[pd.to_datetime(date_str)])
        tasmax_hour = tasmax_hour.expand_dims(time=[pd.to_datetime(date_str)])
        
        # Store BOTH variables
        daily_datasets.append(
            xr.Dataset(
                {
                    "tasmax": tasmax,
                    "tasmax_hour": tasmax_hour,
                }
            )
        )

        ds.close()
        #os.remove(reg_file)
        #os.remove(box_file)

    if not daily_datasets:
        print(f"No valid tasmax files for {year}-{month:02d}")
        return None

    ds_month = xr.concat(daily_datasets, dim="time")
    ds_month.to_netcdf(monthly_file)

    return monthly_file


In [4]:
#test_file = compute_monthly_tasmax_paris_region(
#    year=2021,
#    month=6,
#    hour_path_sf=hour_path_sf,
#    final_path=final_path,
#    lon_min=lon_min,
#    lon_max=lon_max,
#    lat_min=lat_min,
#    lat_max=lat_max,
#)

#print(test_file)


In [10]:
start_date = "2024-01-01"
end_date   = "2025-12-31"

# Create all year–month pairs
months = pd.date_range(start=start_date, end=end_date, freq="MS")

# Keep only JJA (June, July, August)
months_jja = months[months.month.isin([6, 7, 8])]

def run_all_months(hour_path_sf, final_path, n_jobs=50):
    """Parallel runner for compute_monthly_tasmax_paris_region (JJA only)."""

    tasks = [
        delayed(compute_monthly_tasmax_paris_region)(
            date.year,
            date.month,
            hour_path_sf,
            final_path,
            lon_min,
            lon_max,
            lat_min,
            lat_max,
        )
        for date in months_jja
    ]

    # Run in parallel
    Parallel(
        n_jobs=n_jobs,
        verbose=10,
        backend="multiprocessing"
    )(tasks)


In [11]:
run_all_months(
    hour_path_sf,
    final_path,
    n_jobs=40
)


[Parallel(n_jobs=40)]: Using backend MultiprocessingBackend with 40 concurrent workers.
[Parallel(n_jobs=40)]: Done   2 out of   6 | elapsed:   41.5s remaining:  1.4min
[Parallel(n_jobs=40)]: Done   3 out of   6 | elapsed:   42.2s remaining:   42.2s
[Parallel(n_jobs=40)]: Done   4 out of   6 | elapsed:   42.5s remaining:   21.2s
[Parallel(n_jobs=40)]: Done   6 out of   6 | elapsed:   43.0s finished


In [12]:
def process_era5_midlat_box_flexible(year, month, var_num, var,
                                     levels="ml", level="137",
                                     lon_min=0, lon_max=360,
                                     lat_min=40, lat_max=65,
                                     hourly=False,
                                     hourly_agg="tasmax_time",   # NEW: "tasmax_time" or "daily_max"
                                     tasmax_file=None):
    """
    Process ERA5 data over a midlatitude box.

    hourly_agg:
      - "tasmax_time": sample variable at the hour of daily tasmax (needs tasmax_file)
      - "daily_max": compute daily maximum of the variable from hourly data (no tasmax_file needed)
    """
    cdo = Cdo()
    date_str = f"{year}-{month:02d}"

    # Output folders
    var_scratch = os.path.join(scratch_path, var)
    var_final   = os.path.join(final_path, var)
    var_scratch_tmp = os.path.join(var_scratch, "temp/")
    os.makedirs(var_scratch, exist_ok=True)
    os.makedirs(var_final,   exist_ok=True)
    os.makedirs(var_scratch_tmp, exist_ok=True)

    out_file = os.path.join(var_final, f"{var}_{date_str}.nc")

    # --------------------------------------------
    # Hourly workflow
    # --------------------------------------------
    if hourly:
        suffix = "at_tasmax" if hourly_agg == "tasmax_time" else "dailymax"
        out_file = os.path.join(var_final, f"{var}_{date_str}_{suffix}.nc")

        # Only required for tasmax_time mode
        tasmax_ds = None
        if hourly_agg == "tasmax_time":
            tasmax_file = os.path.join(final_path, f"tasmax/tasmax_{year}-{month:02d}_midlatitudes.nc")
            if not os.path.exists(tasmax_file):
                raise ValueError("tasmax_file must exist when hourly_agg='tasmax_time'")
            tasmax_ds = xr.open_dataset(tasmax_file)

        # Determine which ERA5 path to use
        if levels == "ml":
            data_path = hour_path_ml
        elif levels == "pl":
            data_path = hour_path_pl
        else:
            data_path = hour_path_sf

        # Robust month handling
        date_index = pd.date_range(
            start=f"{date_str}-01",
            end=pd.Timestamp(f"{date_str}-01") + pd.offsets.MonthEnd(1)
        )

        daily_datasets = []
        temp_files = []

        # helper to select pressure level robustly
        def _select_level(ds, level_value):
            # Your code uses plev in Pa (e.g. 50000). Keep that, but be robust.
            for cname in ["plev", "isobaricInhPa", "level"]:
                if cname in ds.coords:
                    if cname == "isobaricInhPa":
                        # if ds uses hPa, convert 50000 Pa -> 500 hPa
                        lv = int(level_value) // 100 if int(level_value) > 2000 else int(level_value)
                        return ds.sel({cname: lv})
                    else:
                        return ds.sel({cname: int(level_value)})
            return ds  # fallback: nothing to select

        for day in date_index:
            date_str_h = day.strftime("%Y-%m-%d")
            var_file   = f"{data_path}{var_num}/E5{levels}00_1H_{date_str_h}_{var_num}.grb"

            if not os.path.exists(var_file):
                print(f"Skipping missing: {var_file}")
                continue

            reg_name   = f"reg_{var}_{date_str_h}_hourly.nc"
            box_name   = f"box_{var}_{date_str_h}_hourly.nc"
            out_file_h = os.path.join(var_scratch_tmp, f"{var}_{date_str_h}_midlat.nc")
            reg_file = os.path.join(var_scratch_tmp, reg_name)
            box_file = os.path.join(var_scratch_tmp, box_name)
            temp_files.append(out_file_h)

            # Step 1: GRIB → NetCDF + regular grid
            cdo.setgridtype("regular", input=var_file, output=reg_file, options="-f nc --eccodes")

            # Step 2: Select midlatitude box
            cdo.sellonlatbox(lon_min, lon_max, lat_min, lat_max, input=reg_file, output=box_file)
            os.remove(reg_file)

            # Step 3: Vertical level logic
            if levels == "ml":
                cdo.sellevel(level, input=box_file, output=out_file_h)
                os.remove(box_file)
            elif levels == "pl":
                ds0 = xr.open_dataset(box_file)
                ds0 = _select_level(ds0, level)
                ds0["time"] = pd.to_datetime(ds0["time"].values)
                ds0.to_netcdf(out_file_h)
                ds0.close()
                os.remove(box_file)
            else:
                import shutil
                shutil.move(box_file, out_file_h)

            # ---- Daily reduction from hourly ----
            ds = xr.open_dataset(out_file_h)
            x = ds[var]

            if hourly_agg == "daily_max":
                imax = x.argmax(dim="time")
                vmax = x.isel(time=imax).rename(var)

                # hour of max (optional but handy)
                vhour = x["time"].isel(time=imax).dt.hour.rename(f"{var}_hour")

                day_ts = pd.Timestamp(date_str_h)
                vmax = vmax.reset_coords("time", drop=True).expand_dims(time=[day_ts])
                vhour = vhour.reset_coords("time", drop=True).expand_dims(time=[day_ts])

                daily_datasets.append(xr.Dataset({var: vmax, f"{var}_hour": vhour}))

            elif hourly_agg == "tasmax_time":
                tasmax_hour = tasmax_ds.tasmax_hour.sel(time=date_str_h)

                day_ts = pd.Timestamp(date_str_h)
                tasmax_time = xr.apply_ufunc(
                    lambda h: day_ts + pd.Timedelta(hours=int(h)),
                    tasmax_hour,
                    vectorize=True
                )

                var_at_tasmax = x.sel(time=tasmax_time, method="nearest").rename(var)
                var_at_tasmax = var_at_tasmax.reset_coords("time", drop=True).expand_dims(time=[day_ts])

                daily_datasets.append(xr.Dataset({var: var_at_tasmax}))

            else:
                ds.close()
                raise ValueError("hourly_agg must be 'daily_max' or 'tasmax_time'")

            ds.close()

        if not daily_datasets:
            print(f"No valid files for {year}-{month:02d}")
            if tasmax_ds is not None:
                tasmax_ds.close()
            return None

        ds_month = xr.concat(daily_datasets, dim="time")
        ds_month.to_netcdf(out_file)

        if tasmax_ds is not None:
            tasmax_ds.close()

        for f in temp_files:
            if os.path.exists(f):
                os.remove(f)

        return out_file
        
    # --------------------------------------------
    # Daily workflow (default)
    # --------------------------------------------
    # Determine which ERA5 path to use
    if levels == "ml":
        data_path = data_path_ml
    elif levels == "pl":
        data_path = data_path_pl
    else:
        data_path = day_path_sf   # surface daily fields

    var_file = f"{data_path}{var_num}/E5{levels}00_1D_{date_str}_{var_num}.grb"
    if not os.path.exists(var_file):
        print(f"Missing: {var_file}")
        return

    # Step 1: Convert GRIB → NetCDF + regular grid
    reg_file = os.path.join(var_scratch, f"reg_{var}_{date_str}.nc")
    cdo.setgridtype("regular", input=var_file, output=reg_file, options="-f nc --eccodes")

    # Step 2: Select midlatitude box
    box_file = os.path.join(var_scratch, f"box_{var}_{date_str}.nc")
    cdo.sellonlatbox(lon_min, lon_max, lat_min, lat_max,
                     input=reg_file, output=box_file)

    # Step 3: Vertical level logic
    if levels == "ml":
        cdo.sellevel(level, input=box_file, output=out_file)
        os.remove(box_file)
    elif levels == "pl":
        ds = xr.open_dataset(box_file)
        ds_sel = ds.sel(plev=int(level))
        ds_sel["time"] = pd.to_datetime(ds_sel["time"].values)
        ds_sel.to_netcdf(out_file)
        ds.close()
        os.remove(box_file)
    else:
        import shutil
        shutil.move(box_file, out_file)

    os.remove(reg_file)
    return out_file


In [13]:
import pandas as pd
from joblib import Parallel, delayed

# === Variable mapping ===
era5_vars = {
    134: "sp",
    167: "2t",  # 2m temperature (surface field)
    168: "2d",  # 2m dewpoint temperature (surface field)
    39: "swvl1",
    159: "blh",
    130: "t",   # Temperature @ 500 hPa (pressure level)
    129: "z",   # Geopotential @ 500 hPa (pressure level)
    133: "q",   # Specific humidity @ ML level 137
}

# === Date range (monthly) ===
start_date = "2024-01-01"
end_date   = "2025-12-31"
months = pd.date_range(start_date, end_date, freq="MS")  # monthly start dates

# --- Only JJA ---
months_jja = months[months.month.isin([6, 7, 8])]

# === Wrapper function for both daily and hourly ===
def run_process_daily_hourly(date, var_num, var):
    # level logic
    if var_num in [130, 129]:
        level_type, level = "pl", "50000"
    elif var_num == 133:
        level_type, level = "ml", "137"
    else:
        level_type, level = "sf", None

    # Daily (from 1D product)
    daily_file = process_era5_midlat_box_flexible(
        year=date.year, month=date.month,
        var_num=f"{var_num:03d}", var=var,
        levels=level_type, level=level,
        lon_min=0, lon_max=360, lat_min=40, lat_max=65,
        hourly=False
    )

    # Default: None
    at_tasmax_file = None
    dailymax_file  = None

    if var_num == 130:
        # (A) t at tasmax hour (needs tasmax file produced elsewhere)
        at_tasmax_file = process_era5_midlat_box_flexible(
            year=date.year, month=date.month,
            var_num=f"{var_num:03d}", var=var,
            levels=level_type, level=level,
            lon_min=0, lon_max=360, lat_min=40, lat_max=65,
            hourly=True,
            hourly_agg="tasmax_time"   # <-- sample t at tasmax hour
        )

        # (B) daily max of t from hourly data
        dailymax_file = process_era5_midlat_box_flexible(
            year=date.year, month=date.month,
            var_num=f"{var_num:03d}", var=var,
            levels=level_type, level=level,
            lon_min=0, lon_max=360, lat_min=40, lat_max=65,
            hourly=True,
            hourly_agg="daily_max"     # <-- compute daily max of t
        )
    else:
        # For other vars keep your previous choice (or adapt similarly)
        at_tasmax_file = process_era5_midlat_box_flexible(
            year=date.year, month=date.month,
            var_num=f"{var_num:03d}", var=var,
            levels=level_type, level=level,
            lon_min=0, lon_max=360, lat_min=40, lat_max=65,
            hourly=True,
            hourly_agg="tasmax_time"
        )

    return daily_file, at_tasmax_file, dailymax_file


# === Parallel execution ===
n_jobs = 40

results = Parallel(n_jobs=n_jobs, verbose=10)(
    delayed(run_process_daily_hourly)(month, var_num, var)
    for var_num, var in era5_vars.items()
    for month in months_jja
)

print("Processing completed.")


[Parallel(n_jobs=40)]: Using backend LokyBackend with 40 concurrent workers.
[Parallel(n_jobs=40)]: Done   4 out of  48 | elapsed:  1.7min remaining: 18.2min
[Parallel(n_jobs=40)]: Done   9 out of  48 | elapsed:  1.7min remaining:  7.3min
[Parallel(n_jobs=40)]: Done  14 out of  48 | elapsed:  1.7min remaining:  4.1min
[Parallel(n_jobs=40)]: Done  19 out of  48 | elapsed:  1.7min remaining:  2.6min
[Parallel(n_jobs=40)]: Done  24 out of  48 | elapsed:  1.7min remaining:  1.7min
[Parallel(n_jobs=40)]: Done  29 out of  48 | elapsed:  1.8min remaining:  1.1min
[Parallel(n_jobs=40)]: Done  34 out of  48 | elapsed: 10.2min remaining:  4.2min
[Parallel(n_jobs=40)]: Done  39 out of  48 | elapsed: 18.2min remaining:  4.2min
[Parallel(n_jobs=40)]: Done  44 out of  48 | elapsed: 31.9min remaining:  2.9min


Processing completed.


[Parallel(n_jobs=40)]: Done  48 out of  48 | elapsed: 40.3min finished


In [ ]:
ds = xr.open_dataset("/work/uc1275/u301827/02_MSE/full_midlatitude/raw/2d/2d_1944-06.nc")

In [ ]:
ds["2d"].values